In [ ]:

!pip -q install -U google-genai

from google.colab import userdata

try:
    api_key = userdata.get("GEMINI_API_KEY")
    if api_key:
        print("Gemini API key available ✅")
    else:
        print("Gemini API key not found ❌")
except Exception:
    print("Gemini API key not configured ❌")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 17.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.
Gemini API key not configured ❌


In [ ]:

from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')

if api_key:
    print("API key successfully loaded!")
else:
    print("API key not found. Check Colab Secrets.")


API key successfully loaded!


In [ ]:

from google import genai

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Reply with only: Gemini connection successful."
)

print(response.text)


Gemini connection successful.


In [ ]:

from google.colab import files

uploaded = files.upload()


KeyboardInterrupt: 

In [4]:

from google.colab import files

uploaded = files.upload()


Saving cleaned_mamaearth_data.csv to cleaned_mamaearth_data.csv


In [5]:

import pandas as pd

df = pd.read_csv("cleaned_mamaearth_data.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


Dataset shape: (175, 10)

Column names:
['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'discount_pct', 'payment_method', 'rating', 'returned', 'quantity_outlier']

First 5 rows:


,order_id,customer_id,product_id,order_date,quantity,discount_pct,payment_method,rating,returned,quantity_outlier
0,O0001,C012,P04,2026-03-18,2,10.0,UPI,2.0,0,False
1,O0002,C024,P06,2026-05-16,1,30.0,UPI,1.0,0,False
2,O0003,C008,P12,2026-03-20,2,0.0,COD,1.0,1,False
3,O0004,C032,P03,2026-05-17,3,30.0,UPI,5.0,0,False
4,O0005,C011,P09,2026-06-05,1,10.0,CARD,3.0,0,False


In [6]:

from google.colab import files

uploaded = files.upload()


Saving products.csv to products.csv


In [7]:

from google.colab import files

uploaded = files.upload()


Saving customers.csv to customers.csv


In [8]:

products = pd.read_csv("products.csv")
customers = pd.read_csv("customers.csv")

print("Products shape:", products.shape)
print("Products columns:", products.columns.tolist())

print("\nCustomers shape:", customers.shape)
print("Customers columns:", customers.columns.tolist())


Products shape: (16, 4)
Products columns: ['product_id', 'product_name', 'category', 'price']

Customers shape: (45, 6)
Customers columns: ['customer_id', 'name', 'city', 'city_tier', 'signup_date', 'acquisition_source']


In [9]:

# Merge the cleaned orders with product and customer details
analysis_df = df.merge(products, on="product_id", how="left")
analysis_df = analysis_df.merge(customers, on="customer_id", how="left")

print("Merged dataset shape:", analysis_df.shape)

print("\nMissing product details:", analysis_df["category"].isna().sum())
print("Missing customer details:", analysis_df["city"].isna().sum())

display(analysis_df.head())


Merged dataset shape: (175, 18)

Missing product details: 0
Missing customer details: 0


,order_id,customer_id,product_id,order_date,quantity,discount_pct,payment_method,rating,returned,quantity_outlier,product_name,category,price,name,city,city_tier,signup_date,acquisition_source
0,O0001,C012,P04,2026-03-18,2,10.0,UPI,2.0,0,False,Anti-Hairfall Serum,Haircare,649,Priya,Bangalore,1,2026-01-27,Referral
1,O0002,C024,P06,2026-05-16,1,30.0,UPI,1.0,0,False,Vitamin C Face Wash,Skincare,249,Riya,Lucknow,2,2026-03-04,Referral
2,O0003,C008,P12,2026-03-20,2,0.0,COD,1.0,1,False,Baby Shampoo,Babycare,249,Meera,Lucknow,2,2026-04-18,Ad
3,O0004,C032,P03,2026-05-17,3,30.0,UPI,5.0,0,False,Argan Hair Mask,Haircare,599,Naina,Bangalore,1,2026-02-05,Ad
4,O0005,C011,P09,2026-06-05,1,10.0,CARD,3.0,0,False,Ubtan Face Mask,Skincare,299,Arjun,Bangalore,1,2026-03-13,Referral


In [10]:

# Calculate revenue after discount
analysis_df["revenue"] = (
    analysis_df["quantity"]
    * analysis_df["price"]
    * (1 - analysis_df["discount_pct"] / 100)
)

# Category-wise revenue and return rate
category_summary = analysis_df.groupby("category").agg(
    total_revenue=("revenue", "sum"),
    total_orders=("order_id", "count"),
    returned_orders=("returned", "sum"),
    average_rating=("rating", "mean")
).reset_index()

category_summary["return_rate_pct"] = (
    category_summary["returned_orders"]
    / category_summary["total_orders"] * 100
)

print("Category-wise analysis:")
display(category_summary.round(2))

print("\nTotal revenue:", round(analysis_df["revenue"].sum(), 2))
print("Overall return rate (%):",
      round(analysis_df["returned"].mean() * 100, 2))


Category-wise analysis:


,category,total_revenue,total_orders,returned_orders,average_rating,return_rate_pct
0,Babycare,16535.9,29,7,2.66,24.14
1,Haircare,43698.2,53,14,3.04,26.42
2,PersonalCare,10190.4,34,5,3.15,14.71
3,Skincare,26933.8,59,18,3.10,30.51



Total revenue: 97358.3
Overall return rate (%): 25.14


In [11]:

analysis_df["order_date"] = pd.to_datetime(analysis_df["order_date"])

analysis_df["month"] = (
    analysis_df["order_date"].dt.to_period("M").astype(str)
)

monthly_revenue = (
    analysis_df.groupby("month")["revenue"]
    .sum()
    .reset_index()
)

monthly_revenue["revenue"] = monthly_revenue["revenue"].round(2)

print("Monthly revenue analysis:")
display(monthly_revenue)

print("\nHighest revenue month:")
display(monthly_revenue.loc[monthly_revenue["revenue"].idxmax()])


Monthly revenue analysis:


,month,revenue
0,2026-01,29582.1
1,2026-02,13195.5
2,2026-03,20318.9
3,2026-04,9495.3
4,2026-05,13151.1
5,2026-06,11615.4



Highest revenue month:


,0
month,2026-01
revenue,29582.1


In [12]:

import json

findings = {
    "total_revenue": round(float(analysis_df["revenue"].sum()), 2),
    "total_orders": int(len(analysis_df)),
    "overall_return_rate_pct": round(
        float(analysis_df["returned"].mean() * 100), 2
    ),
    "category_analysis": category_summary.to_dict(orient="records"),
    "monthly_revenue": monthly_revenue.to_dict(orient="records"),
    "highest_revenue_month": str(
        monthly_revenue.loc[monthly_revenue["revenue"].idxmax(), "month"]
    ),
    "highest_revenue_category": str(
        category_summary.loc[
            category_summary["total_revenue"].idxmax(), "category"
        ]
    ),
    "highest_return_rate_category": str(
        category_summary.loc[
            category_summary["return_rate_pct"].idxmax(), "category"
        ]
    )
}

with open("findings.json", "w") as f:
    json.dump(findings, f, indent=4)

print("findings.json created successfully!")
print(json.dumps(findings, indent=2))


findings.json created successfully!
{
  "total_revenue": 97358.3,
  "total_orders": 175,
  "overall_return_rate_pct": 25.14,
  "category_analysis": [
    {
      "category": "Babycare",
      "total_revenue": 16535.9,
      "total_orders": 29,
      "returned_orders": 7,
      "average_rating": 2.6551724137931036,
      "return_rate_pct": 24.137931034482758
    },
    {
      "category": "Haircare",
      "total_revenue": 43698.2,
      "total_orders": 53,
      "returned_orders": 14,
      "average_rating": 3.0377358490566038,
      "return_rate_pct": 26.41509433962264
    },
    {
      "category": "PersonalCare",
      "total_revenue": 10190.4,
      "total_orders": 34,
      "returned_orders": 5,
      "average_rating": 3.1470588235294117,
      "return_rate_pct": 14.705882352941178
    },
    {
      "category": "Skincare",
      "total_revenue": 26933.8,
      "total_orders": 59,
      "returned_orders": 18,
      "average_rating": 3.1016949152542375,
      "return_rate_pct": 30.

In [13]:

import json

with open("findings.json", "r") as f:
    findings = json.load(f)

prompt = f"""
You are a data analyst working on a Mamaearth e-commerce capstone project.

Write a clear business analysis narrative based ONLY on the verified
data provided below.

Include:
1. Overall revenue and number of orders.
2. Category-wise revenue and return-rate insights.
3. Monthly revenue trends.
4. Three practical business recommendations.

Rules:
- Use only the supplied data.
- Do not invent causes, facts, or statistics.
- Clearly distinguish observations from possible explanations.
- Keep the language simple and professional.
- Mention that the data covers January to June 2026.

Verified findings:
{json.dumps(findings, indent=2)}
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

narrative = response.text

print(narrative)


### Business Analysis Report: Mamaearth E-commerce (January – June 2026)

This report provides a data-driven overview of Mamaearth’s business performance for the first half of 2026.

#### 1. Overall Performance
Between January and June 2026, the platform generated a **total revenue of 97,358.3** across **175 total orders**. The business recorded an **overall return rate of 25.14%** for this period.

#### 2. Category-wise Revenue and Return Analysis
The following table summarizes the performance metrics by product category:

| Category | Revenue | Orders | Return Rate | Avg. Rating |
| :--- | :--- | :--- | :--- | :--- |
| **Haircare** | 43,698.2 | 53 | 26.41% | 3.04 |
| **Skincare** | 26,933.8 | 59 | 30.51% | 3.10 |
| **Babycare** | 16,535.9 | 29 | 24.14% | 2.66 |
| **PersonalCare** | 10,190.4 | 34 | 14.71% | 3.15 |

*   **Key Observations:**
    *   **Haircare** is the highest revenue-generating category.
    *   **Skincare** accounts for the highest volume of orders (59) but also suff

In [14]:

# Verify key numbers in the Gemini-generated narrative
checks = {
    "Total revenue": "97,358.3",
    "Total orders": "175",
    "Overall return rate": "25.14%",
    "Haircare revenue": "43,698.2",
    "Skincare return rate": "30.51%",
    "January revenue": "29,582.1",
    "April revenue": "9,495.3"
}

print("NARRATIVE ACCURACY CHECK")
print("-" * 35)

all_passed = True

for label, expected in checks.items():
    passed = expected in narrative
    print(f"{label}: {'PASS' if passed else 'CHECK'}")
    if not passed:
        all_passed = False

print("-" * 35)
print("Overall result:", "ALL CHECKS PASSED" if all_passed
      else "Some values need review")


NARRATIVE ACCURACY CHECK
-----------------------------------
Total revenue: PASS
Total orders: PASS
Overall return rate: PASS
Haircare revenue: PASS
Skincare return rate: PASS
January revenue: PASS
April revenue: PASS
-----------------------------------
Overall result: ALL CHECKS PASSED


In [15]:
with open("genai_narrative.txt", "w", encoding="utf-8") as f:
    f.write(narrative)

print("Actual Gemini narrative saved as genai_narrative.txt")

Actual Gemini narrative saved as genai_narrative.txt


In [16]:
from google.colab import files

files.download("findings.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
files.download("genai_narrative.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>